# 03. ML Agent — 머신러닝 Tool 통합
**Day 14**

> Day 8~11에서 배운 머신러닝 모델을 **LangChain `@tool`** 로 감싸 Agent가 호출합니다.  
> 구현 코드: `ml_tools/` 폴더 (로지스틱 회귀 · XGBoost · One-Class SVM)

---
## 0. 설치 & 경로

In [1]:
!pip install openai python-dotenv langchain langchain-openai langchain-core langchain-classic scikit-learn xgboost joblib

In [2]:
import sys
from pathlib import Path

from dotenv import load_dotenv

load_dotenv()
OPENAI_API_KEY = __import__('os').getenv('OPENAI_API_KEY')

WORKDIR = Path.cwd()
ML_TOOLS_DIR = WORKDIR / 'ml_tools'
if str(WORKDIR) not in sys.path:
    sys.path.insert(0, str(WORKDIR))

print('WORKDIR:', WORKDIR)
print('ML_TOOLS:', ML_TOOLS_DIR)
for p in sorted((ML_TOOLS_DIR / 'data').glob('*.csv')):
    print(' - data/', p.name)

WORKDIR: c:\Users\Admin\Desktop\실습폴더\14일차_실습
ML_TOOLS: c:\Users\Admin\Desktop\실습폴더\14일차_실습\ml_tools
 - data/ anomaly_sensor.csv
 - data/ failure_equipment.csv


---
## 1. Day 8~11 ML 방법론 요약

| Tool | 강의 | 알고리즘 | Agent가 쓰는 상황 |
|------|------|----------|-------------------|
| `predict_failure_logistic` | 로지스틱 회귀 | 설비 고장 Yes/No + **확률** (해석·설명) |
| `predict_failure_xgboost` | XGBoost | 불균형 고장 데이터 **예측 성능** 우선 |
| `detect_anomaly_oneclass` | One-Class SVM | 정상 기준 **이상 탐지** (2D 센서) |

공통 패턴:
1. `ml_tools/`에서 **사전 학습 모델** 로드 (`models/*.joblib`)
2. `@tool` 함수가 **문자열**로 결과 반환 → Agent가 읽고 답변

---
## Step 1. ML Tool 불러오기

In [3]:
from ml_tools import ML_TOOLS, predict_failure_logistic, predict_failure_xgboost, detect_anomaly_oneclass

print('등록된 ML Tool:')
for t in ML_TOOLS:
    print(f' - {t.name}: {t.description[:60]}...')

등록된 ML Tool:
 - predict_failure_logistic: 설비 고장(Failure) 여부를 로지스틱 회귀(Day 2-2)로 예측합니다.

Args:
    tempe...
 - predict_failure_xgboost: 설비 고장(Failure) 여부를 XGBoost(Day 3-2)로 예측합니다.

Args:
    tempe...
 - detect_anomaly_oneclass: 2차원 센서값이 정상 범위인지 One-Class SVM(Day 4-3 / SVDD)으로 탐지합니다.

Arg...


---
## ✏️ 실습 1. `@tool` 직접 만들기

**목표:** 현재 시각을 `HH:MM` 형식으로 반환하는 `get_time_hhmm` Tool을 만듭니다.

**힌트:** `from langchain_core.tools import tool` · `datetime.now().strftime('%H:%M')`

In [4]:
# ── 여기에 작성 ──
from datetime import datetime
from langchain_core.tools import tool

@tool
def get_time_hhmm() -> str:
    """현재 시각을 HH:MM 형식으로 반환합니다."""
    return datetime.now().strftime('%H:%M')

print(get_time_hhmm.invoke({}))

16:47


---
## Step 2. Tool 단독 테스트

Agent 없이 `.invoke()`로 먼저 확인합니다. (14.1의 `get_current_time.invoke`와 동일)

In [14]:
# 설비 고장 예측 — 로지스틱 회귀 (Day 2-2)
print(predict_failure_logistic.invoke({
    'temperature': 67,
    'humidity': 82,
    'operator': 'Operator1',
    'hours_since_previous_failure': 90,
}))

[로지스틱 회귀] 예측: No
- 고장(Failure) 확률: 1.98%
- 정상(No) 확률: 98.02%


In [6]:
# 설비 고장 예측 — XGBoost (Day 3-2)
print(predict_failure_xgboost.invoke({
    'temperature': 67,
    'humidity': 82,
    'operator': 'Operator1',
    'hours_since_previous_failure': 90,
}))

[XGBoost] 예측: No
- 고장(Failure) 확률: 0.74%
- 정상(No) 확률: 99.26%
- 주요 변수(학습 기준): Temperature(0.217), Hours Since Previous Failure(0.138), Humidity(0.094)


In [7]:
# 이상 탐지 — One-Class SVM (Day 4-3 / SVDD)
samples = [(1.0, 1.5, '정상에 가까운 값'), (5.0, -4.0, '이상에 가까운 값')]
for f0, f1, label in samples:
    print(f'[{label}]')
    print(detect_anomaly_oneclass.invoke({'feature_0': f0, 'feature_1': f1}))
    print('-' * 40)

[정상에 가까운 값]
[One-Class SVM] 판정: 이상
- decision_score: -0.0711
- raw_prediction: -1 (+1 정상 / -1 이상)
----------------------------------------
[이상에 가까운 값]
[One-Class SVM] 판정: 이상
- decision_score: -0.9268
- raw_prediction: -1 (+1 정상 / -1 이상)
----------------------------------------


c:\Users\Admin\anaconda3\Lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator OneClassSVM from version 1.9.0 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\Admin\anaconda3\Lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator OneClassSVM from version 1.9.0 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


---
## 센서값 직접 바꿔 `.invoke()`

`detect_anomaly_oneclass`에 임의의 `feature_0`, `feature_1` 값을 넣어 결과를 확인하세요.

**관찰:** decision_score가 0에 가까울수록 경계선 샘플입니다.

In [ ]:
# ── 여기에 작성 ──
# my_f0 = 0.5
# my_f1 = 1
# print(detect_anomaly_oneclass.invoke({'feature_0': my_f0, 'feature_1': my_f1}))

[One-Class SVM] 판정: 이상
- decision_score: -0.4037
- raw_prediction: -1 (+1 정상 / -1 이상)


c:\Users\Admin\anaconda3\Lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator OneClassSVM from version 1.9.0 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


---
## Step 3. AgentExecutor — ML Tool 통합

14.1과 같이 `create_tool_calling_agent` + `AgentExecutor`를 씁니다.

In [13]:
from langchain_openai import ChatOpenAI
from langchain_classic.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

llm = ChatOpenAI(model='gpt-4o-mini', temperature=0.1)

agent_tools = ML_TOOLS
print([t.name for t in agent_tools])

prompt = ChatPromptTemplate.from_messages([
    ('system', '''당신은 제조 설비 분석을 돕는 AI입니다. 질문에 맞는 ML 도구를 골라 사용하세요.

- 설비 고장(Failure) 확률·해석 → predict_failure_logistic
- 설비 고장 예측 (XGBoost, 불균형 데이터) → predict_failure_xgboost
- 2D 센서값 정상/이상 판정 → detect_anomaly_oneclass

온도·습도·작업자·이전 고장 경과시간이 주어지면 ML 예측 도구를 사용하세요.
센서 feature_0, feature_1 값이 주어지면 이상 탐지 도구를 사용하세요.
도구 결과에 없으면 추측하지 마세요. 한국어로 답하세요.'''),
    MessagesPlaceholder('chat_history', optional=True),
    ('human', '{input}'),
    MessagesPlaceholder('agent_scratchpad'),
])

agent = create_tool_calling_agent(llm, agent_tools, prompt)
executor = AgentExecutor(agent=agent, tools=agent_tools, verbose=True, max_iterations=8)

['predict_failure_logistic', 'predict_failure_xgboost', 'detect_anomaly_oneclass']


---
## ✏️ 실습 3. 시스템 프롬프트 수정

Step 3의 `prompt` **system** 문자열에 아래 규칙을 **한 줄** 추가한 뒤, Agent를 다시 만들어 보세요.

> *"로지스틱과 XGBoost 결과를 비교해 달라고 하면 두 Tool을 모두 호출하세요."*

**힌트:** `agent` · `executor` 변수를 다시 할당하면 됩니다.

In [ ]:
# ── 여기에 작성 ──
# (Step 3 prompt 복사 후 system 문자열에 규칙 1줄 추가)
# agent = create_tool_calling_agent(llm, agent_tools, prompt)
# executor = AgentExecutor(agent=agent, tools=agent_tools, verbose=True, max_iterations=8)

---
## Step 4. Agent 데모 질문

In [15]:
# 로지스틱 회귀 Tool 선택 유도
r = executor.invoke({
    'input': '온도 67, 습도 82, Operator1, 이전 고장 후 90시간 — 로지스틱으로 고장 확률 알려줘',
    'chat_history': [],
})
print(r['output'])



> Entering new AgentExecutor chain...

Invoking: `predict_failure_logistic` with `{'temperature': 67, 'humidity': 82, 'operator': 'Operator1', 'hours_since_previous_failure': 90}`


[로지스틱 회귀] 예측: No
- 고장(Failure) 확률: 1.98%
- 정상(No) 확률: 98.02%로지스틱 회귀 분석 결과, 고장 확률은 1.98%로 나타났습니다. 정상 확률은 98.02%입니다.

> Finished chain.
로지스틱 회귀 분석 결과, 고장 확률은 1.98%로 나타났습니다. 정상 확률은 98.02%입니다.


In [16]:
# XGBoost Tool 선택 유도
r = executor.invoke({
    'input': '같은 조건으로 XGBoost 모델 예측도 해줘',
    'chat_history': [],
})
print(r['output'])



> Entering new AgentExecutor chain...

Invoking: `predict_failure_xgboost` with `{'temperature': 67, 'humidity': 82, 'operator': 'Operator1', 'hours_since_previous_failure': 5}`


[XGBoost] 예측: No
- 고장(Failure) 확률: 0.86%
- 정상(No) 확률: 99.14%
- 주요 변수(학습 기준): Temperature(0.217), Hours Since Previous Failure(0.138), Humidity(0.094)XGBoost 모델 예측 결과는 다음과 같습니다:

- 고장(Failure) 예측: **아니오 (No)**
- 고장 확률: **0.86%**
- 정상 확률: **99.14%**
- 주요 변수(학습 기준):
  - 온도(Temperature): 0.217
  - 이전 고장 경과 시간(Hours Since Previous Failure): 0.138
  - 습도(Humidity): 0.094

따라서, 현재 조건에서는 설비 고장이 발생할 가능성이 매우 낮습니다.

> Finished chain.
XGBoost 모델 예측 결과는 다음과 같습니다:

- 고장(Failure) 예측: **아니오 (No)**
- 고장 확률: **0.86%**
- 정상 확률: **99.14%**
- 주요 변수(학습 기준):
  - 온도(Temperature): 0.217
  - 이전 고장 경과 시간(Hours Since Previous Failure): 0.138
  - 습도(Humidity): 0.094

따라서, 현재 조건에서는 설비 고장이 발생할 가능성이 매우 낮습니다.


In [17]:
# One-Class SVM 이상 탐지
r = executor.invoke({
    'input': '센서 feature_0=1.0, feature_1=1.5 일 때 정상인지 이상인지 판정해줘',
    'chat_history': [],
})
print(r['output'])



> Entering new AgentExecutor chain...

Invoking: `detect_anomaly_oneclass` with `{'feature_0': 1.0, 'feature_1': 1.5}`


[One-Class SVM] 판정: 이상
- decision_score: -0.0711
- raw_prediction: -1 (+1 정상 / -1 이상)

c:\Users\Admin\anaconda3\Lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator OneClassSVM from version 1.9.0 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


센서 feature_0=1.0, feature_1=1.5일 때 판정 결과는 "이상"입니다.

> Finished chain.
센서 feature_0=1.0, feature_1=1.5일 때 판정 결과는 "이상"입니다.


---
## ✏️ 실습 4. Agent에 본인 질문 작성

**고장 예측** 또는 **이상 탐지** 중 하나를 골라, Step 4 데모와 다른 조건으로 `executor.invoke`를 직접 호출하세요.

**예시:** *"온도 80, 습도 90, Operator3, 이전 고장 10시간 — XGBoost로 예측해줘"*

In [18]:
# ── 여기에 작성 ──
my_question = '습도 90, Oper 1, 이전 고장 한시간 예측, 온도는 모름'
r = executor.invoke({'input': my_question, 'chat_history': []})
print(r['output'])



> Entering new AgentExecutor chain...

Invoking: `predict_failure_logistic` with `{'temperature': 67, 'humidity': 90, 'operator': 'Operator1', 'hours_since_previous_failure': 1}`


[로지스틱 회귀] 예측: No
- 고장(Failure) 확률: 0.96%
- 정상(No) 확률: 99.04%예측 결과는 "고장 없음(No)"입니다. 고장 확률은 0.96%로 매우 낮고, 정상 확률은 99.04%입니다.

> Finished chain.
예측 결과는 "고장 없음(No)"입니다. 고장 확률은 0.96%로 매우 낮고, 정상 확률은 99.04%입니다.


---
## Step 5. 멀티턴 — `chat_history`

In [19]:
from langchain_core.messages import HumanMessage, AIMessage

class ChatSession:
    def __init__(self, executor):
        self.executor = executor
        self.history = []

    def ask(self, question: str) -> str:
        result = self.executor.invoke({'input': question, 'chat_history': self.history})
        answer = result['output']
        self.history.extend([HumanMessage(content=question), AIMessage(content=answer)])
        return answer

session = ChatSession(executor)
print('1:', session.ask('온도 70, 습도 75, Operator2, 이전 고장 120시간 — 고장 날까?'))
print('2:', session.ask('방금 답에서 고장 확률만 숫자로 다시 말해줘'))



> Entering new AgentExecutor chain...

Invoking: `predict_failure_logistic` with `{'temperature': 70, 'humidity': 75, 'operator': 'Operator2', 'hours_since_previous_failure': 120}`


[로지스틱 회귀] 예측: No
- 고장(Failure) 확률: 11.09%
- 정상(No) 확률: 88.91%예측 결과에 따르면, 고장(Failure) 확률은 11.09%로 낮으며, 정상(No) 확률은 88.91%입니다. 따라서 고장이 날 가능성은 낮습니다.

> Finished chain.
1: 예측 결과에 따르면, 고장(Failure) 확률은 11.09%로 낮으며, 정상(No) 확률은 88.91%입니다. 따라서 고장이 날 가능성은 낮습니다.


> Entering new AgentExecutor chain...
고장 확률은 11.09%입니다.

> Finished chain.
2: 고장 확률은 11.09%입니다.


---
## ✏️ 실습 5. 멀티턴 후속 질문

`ChatSession`으로 1턴 예측 후, **2턴에서 맥락을 이어가는** 후속 질문을 직접 작성하세요.

**예시:** 1턴 고장 예측 → 2턴 *"그럼 같은 조건으로 이상 탐지도 해줘 feature_0=2, feature_1=2"*

In [21]:
# ── 여기에 작성 ──
session2 = ChatSession(executor)
q1 = '온도 70, 습도 75, Operator2, 이전 고장 120시간, 고장 확률은?'
q2 = 'feature_0=2, feature_1=2, 같은 조건으로 이상 탐지 해줘.'  # 1턴 답을 참조하는 후속 질문
print('1:', session2.ask(q1))
print('2:', session2.ask(q2))



> Entering new AgentExecutor chain...

Invoking: `predict_failure_logistic` with `{'temperature': 70, 'humidity': 75, 'operator': 'Operator2', 'hours_since_previous_failure': 120}`


[로지스틱 회귀] 예측: No
- 고장(Failure) 확률: 11.09%
- 정상(No) 확률: 88.91%예측 결과는 "고장 없음(No)"입니다. 고장 확률은 11.09%이며, 정상 확률은 88.91%입니다.

> Finished chain.
1: 예측 결과는 "고장 없음(No)"입니다. 고장 확률은 11.09%이며, 정상 확률은 88.91%입니다.


> Entering new AgentExecutor chain...

Invoking: `detect_anomaly_oneclass` with `{'feature_0': 2, 'feature_1': 2}`


[One-Class SVM] 판정: 정상
- decision_score: 0.0533
- raw_prediction: 1 (+1 정상 / -1 이상)

c:\Users\Admin\anaconda3\Lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator OneClassSVM from version 1.9.0 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


이상 탐지 결과는 "정상"입니다. decision_score는 0.0533으로, 정상에 가깝습니다.

> Finished chain.
2: 이상 탐지 결과는 "정상"입니다. decision_score는 0.0533으로, 정상에 가깝습니다.


---
## Step 6. (선택) 14.1 Agent와 합치기

RAG · 웹 검색 · ML을 한 Agent에 넣으려면 `agent_tools` 리스트에 합치면 됩니다.

### ✏️ 실습 6

1. **14.1 노트북에 정의**된 `get_current_time`, `search_pdf_tool`, `web_search`코드를 복사해 선언합니다.
2. 아래 셀에서 `agent_tools`를 채우고, `system` 프롬프트에 RAG·웹·ML Tool 용도를 각각 한 줄씩 추가하세요.
3. (선택) 합친 Agent로 *"석사 수업연한이 몇 학기야?"* 와 *"온도 70 습도 80 고장 예측"* 을 연속 질문해 보세요.

In [26]:
import os
from datetime import datetime
from pathlib import Path

import pytz
from langchain_chroma import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.tools import DuckDuckGoSearchResults
from langchain_core.tools import create_retriever_tool, tool
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

@tool
def get_current_time(timezone: str, location: str) -> str:
    """현재 시각. timezone 예: Asia/Seoul, location 예: 서울"""
    tz = pytz.timezone(timezone)
    now = datetime.now(tz).strftime('%Y-%m-%d %H:%M:%S')
    return f'{timezone} ({location}) 현재시각 {now}'

web_search = DuckDuckGoSearchResults(num_results=3)

# RAG (14.1과 동일)
RAG_SAMPLES = Path(os.getcwd()) / 'samples' / 'pdf_samples'
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)

loader = PyPDFLoader(str(RAG_SAMPLES / '학칙.pdf'))
all_splits = text_splitter.split_documents(loader.load())

loader_news = PyPDFLoader(str(RAG_SAMPLES / '보도자료.pdf'))
all_splits.extend(text_splitter.split_documents(loader_news.load()))

embedding = OpenAIEmbeddings(model='text-embedding-3-large', api_key=OPENAI_API_KEY)
persist_directory = 'chroma_store'

if not os.path.exists(persist_directory):
    print('Creating new Chroma store')
    vectorstore = Chroma.from_documents(
        documents=all_splits,
        embedding=embedding,
        persist_directory=persist_directory,
    )
else:
    print('Loading existing Chroma store')
    vectorstore = Chroma(
        persist_directory=persist_directory,
        embedding_function=embedding,
    )

retriever = vectorstore.as_retriever(search_kwargs={'k': 3})

search_pdf_tool = create_retriever_tool(
    retriever,
    name='search_pdf_documents',
    description='pdf samples 안의 문서함에서 질문과 관련된 내용을 검색합니다.',
)

Loading existing Chroma store


In [27]:
agent_tools = [
    get_current_time,
    search_pdf_tool,
    web_search,
    *ML_TOOLS,
]

combined_prompt = ChatPromptTemplate.from_messages([
    ('system', '''당신은 사용자를 돕는 AI입니다. 질문에 맞는 도구를 골라 사용하세요.

- samples/pdf_samples PDF (학칙, 보도자료 등) → search_pdf_documents
- 최신 뉴스·시사·웹 정보 → duckduckgo_results_json
- 설비 고장(Failure) 확률·해석 → predict_failure_logistic
- 설비 고장 예측 (XGBoost) → predict_failure_xgboost
- 2D 센서값 정상/이상 판정 → detect_anomaly_oneclass
- 현재 시각 → get_current_time

도구 결과에 없으면 추측하지 마세요. 한국어로 답하세요.'''),
    MessagesPlaceholder('chat_history', optional=True),
    ('human', '{input}'),
    MessagesPlaceholder('agent_scratchpad'),
])

combined_agent = create_tool_calling_agent(llm, agent_tools, combined_prompt)
combined_executor = AgentExecutor(agent=combined_agent, tools=agent_tools, verbose=True, max_iterations=8)
print([t.name for t in agent_tools])

['get_current_time', 'search_pdf_documents', 'duckduckgo_results_json', 'predict_failure_logistic', 'predict_failure_xgboost', 'detect_anomaly_oneclass']


---
## Step 7. `input()` 대화 루프

In [28]:
print('종료: exit')
while True:
    user_input = input('사용자: ').strip()
    if user_input.lower() in ('exit', 'quit', '종료'):
        break
    if not user_input:
        continue
    print('AI:', session.ask(user_input))

종료: exit


> Entering new AgentExecutor chain...
석사 수업 연한은 일반적으로 2년입니다. 그러나 대학이나 전공에 따라 다를 수 있으니, 구체적인 사항은 해당 대학의 규정을 확인하는 것이 좋습니다.

> Finished chain.
AI: 석사 수업 연한은 일반적으로 2년입니다. 그러나 대학이나 전공에 따라 다를 수 있으니, 구체적인 사항은 해당 대학의 규정을 확인하는 것이 좋습니다.


> Entering new AgentExecutor chain...

Invoking: `predict_failure_xgboost` with `{'temperature': 79, 'humidity': 80, 'operator': 'Operator1', 'hours_since_previous_failure': 0}`


[XGBoost] 예측: Yes
- 고장(Failure) 확률: 95.93%
- 정상(No) 확률: 4.07%
- 주요 변수(학습 기준): Temperature(0.217), Hours Since Previous Failure(0.138), Humidity(0.094)예측 결과에 따르면, 고장(Failure) 확률은 95.93%로 매우 높습니다. 따라서 고장이 날 가능성이 큽니다.

> Finished chain.
AI: 예측 결과에 따르면, 고장(Failure) 확률은 95.93%로 매우 높습니다. 따라서 고장이 날 가능성이 큽니다.


---
## 정리

| 파일 | 역할 |
|------|------|
| `ml_tools/logistic_regression_tool.py` | Day 9 로지스틱 회귀 |
| `ml_tools/xgboost_classifier_tool.py` | Day 10 XGBoost |
| `ml_tools/oneclass_svm_tool.py` | Day 11 One-Class SVM |
| `ml_tools/models/*.joblib` | 사전 학습 모델 (자동 생성) |

> ML Tool은 **추론만** 수행합니다. 재학습이 필요하면 각 `.py`를 `python logistic_regression_tool.py`처럼 직접 실행하세요.